In [1]:
%load_ext autoreload
%autoreload 2
from pc_layers import PredictiveCodingNetwork
from torch.nn import functional as F
import torch
from dataloader import TraceDataset
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
def evaluate_pcn(model, data_loader, device='cuda', T_infer=20, eta_infer=0.1):
    model.to(device).eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in data_loader:
            B = batch['features'].size(0)
            d_0 = model.dims[0]
            x_batch = batch['features'].view(B, d_0).to(device)
            y_batch = F.one_hot(batch['index'], num_classes=model.readout.out_features).float().to(device)
            inputs_latents = [x_batch] + model.init_latents(B, device)
            weights = [layer.W for layer in model.layers] + [model.readout.weight]
            # INFERENCE - T_infer steps
            for t in range(1, T_infer + 1):
                errors, gain_modulated_errors = model.compute_errors(inputs_latents)
                y_hat = model.readout(inputs_latents[-1])
                eps_sup = y_hat - y_batch
                eps_L = eps_sup @ weights[-1]
                errors_extended = errors + [eps_L]
                # Latent gradients and updates
                for l in range(1, model.L + 1):
                    grad_Xl = errors_extended[l] - gain_modulated_errors[l-1] @ weights[l-1]
                    inputs_latents[l] -= eta_infer * grad_Xl
            # Collect predictions
            final_y_hat = model.readout(inputs_latents[-1])
            preds = final_y_hat.argmax(dim=1).cpu().numpy()
            labels = batch['index'].cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels)
    return np.concatenate(all_preds), np.concatenate(all_labels)

In [29]:
def train_pcn(model, data_loader, num_epochs, eta_infer, eta_learn,
T_infer, T_learn, device='cuda'):
    model.to(device).train()
    pbar = tqdm(total=num_epochs * len(data_loader), desc="Training PCN")
    for epoch in range(num_epochs):
        for batch in data_loader:
            B = batch['features'].size(0)
            d_0 = model.dims[0]
            x_batch = batch['features'].view(B, d_0).to(device)
            y_batch = F.one_hot(batch['index'], num_classes=model.readout.out_features).float().to(device)
            inputs_latents = [x_batch] + model.init_latents(B, device)
            weights = [layer.W for layer in model.layers] + [model.readout.weight]
            # INFERENCE - T_infer steps
            with torch.no_grad():
                for t in range(1, T_infer + 1):
                    errors, gain_modulated_errors = model.compute_errors(inputs_latents)
                    y_hat = model.readout(inputs_latents[-1])
                    eps_sup = y_hat - y_batch
                    eps_L = eps_sup @ weights[-1]
                    errors_extended = errors + [eps_L]
                    # Latent gradients and updates
                    for l in range(1, model.L + 1):
                        grad_Xl = errors_extended[l] - gain_modulated_errors[l-1] @ weights[l-1]
                        inputs_latents[l] -= eta_infer * grad_Xl
            # LEARNING - T_learn steps
            with torch.no_grad():
                for t in range(T_infer + 1, T_learn + T_infer + 1):
                    errors, gain_modulated_errors = model.compute_errors(inputs_latents)
                    y_hat = model.readout(inputs_latents[-1])
                    eps_sup = y_hat - y_batch
                    # Weight gradients and updates
                    for l in range(model.L):
                        grad_Wl = -(gain_modulated_errors[l].T @ inputs_latents[l+1]) / B
                        weights[l] -= eta_learn * grad_Wl
                    grad_Wout = eps_sup.T @ inputs_latents[-1] / B
                    weights[-1] -= eta_learn * grad_Wout
            preds, labels = evaluate_pcn(model, data_loader, device, T_infer, eta_infer)
            pbar.set_postfix({"Epoch" : {epoch+1}, "Acc": f"{accuracy_score(labels, preds):.4f}"})
            pbar.update(1)

In [30]:
train_dataset = TraceDataset(mean_approx=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=215, shuffle=True)
dims = [63, 128]
output_dim = 215
model = PredictiveCodingNetwork(dims, 215)

/home/egor/Documents/Projects/Msc thesis/trace_python/predcoding/dataloader.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features.append(torch.tensor(word_feat, dtype=torch.float32))


In [31]:
train_pcn(model, train_loader, num_epochs=15, eta_infer=0.1, eta_learn=0.01, T_infer=70, T_learn=215)

Training PCN: 100%|██████████| 15/15 [00:01<00:00, 12.03it/s, Epoch={15}, Acc=1.0000]


In [38]:
preds, labels = evaluate_pcn(model, train_loader, device='cuda', T_infer=70, eta_infer=0.1)

In [39]:
accuracy = accuracy_score(labels, preds)
print(f"Training Accuracy: {accuracy:.4f}")

Training Accuracy: 1.0000
